<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Her session başında bunu çalıştır
!git clone https://github.com/llelus/DSA-Project.git
%cd DSA-Project
!pip install ccxt -q
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/llelus/DSA-Project.git

Cloning into 'DSA-Project'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 177 (delta 53), reused 115 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 3.00 MiB | 10.30 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/DSA-Project/DSA-Project


In [13]:
import requests
import pandas as pd
import ccxt
import time
import ast
import os
from datetime import datetime, timedelta, timezone

In [ ]:
all_btc = []
offset  = 0

SEARCH_TERMS = [
    "Bitcoin above", "Bitcoin Below",
    "BTC above", "BTC below",
    "Bitcoin Up or Down", "BTC Up or Down",
    "Bitcoin end", "BTC price",
]

while offset < 5000:
    r = requests.get("https://gamma-api.polymarket.com/markets", params={
        "closed":    "true",
        "limit":     100,
        "offset":    offset,
        "order":     "startDate",
        "ascending": "false",
    })
    data = r.json()
    if not isinstance(data, list) or len(data) == 0:
        break

    btc = [
        m for m in data
        if isinstance(m, dict) and
        any(term in m.get("question", "") for term in SEARCH_TERMS)
    ]
    all_btc.extend(btc)
    offset += 100
    time.sleep(0.2)

print(f"Total Bitcoin markets found: {len(all_btc)}")
if all_btc:
    for m in all_btc[:5]:
        print(f"  {m.get('question')} | {m.get('startDate', '')[:10]}")


In [16]:
df_poly_clean = pd.DataFrame(all_btc)
df_poly_clean["startDate"] = pd.to_datetime(df_poly_clean["startDate"], utc=True)
df_poly_clean["endDate"] = pd.to_datetime(df_poly_clean["endDate"], utc=True)
df_poly_clean["yes_price_open"] = df_poly_clean["outcomePrices"].apply(
    lambda x: float(eval(x)[0]) if isinstance(x, str) else float(x[0])
)
df_poly_clean["result_yes_won"] = df_poly_clean["outcomePrices"].apply(
    lambda x: float(eval(x)[0]) if isinstance(x, str) else float(x[0])
).apply(lambda p: 1 if p == 1.0 else 0)

df_poly_clean = df_poly_clean.drop_duplicates(subset="conditionId").reset_index(drop=True)
print(f"Unique market: {len(df_poly_clean)}")
print(df_poly_clean["question"].tolist()[:5])

Unique market: 40
['Bitcoin above 78,000 on May 4, 3AM ET?', 'Bitcoin above 79,200 on May 4, 3AM ET?', 'Bitcoin above 81,600 on May 4, 3AM ET?', 'Bitcoin above 80,400 on May 4, 3AM ET?', 'Bitcoin above 78,800 on May 4, 3AM ET?']


In [17]:
all_price_history = []

for idx, row in df_poly_clean.iterrows():
    token_id = ast.literal_eval(row["clobTokenIds"])[0]

    r = requests.get("https://clob.polymarket.com/prices-history", params={
        "market":   token_id,
        "interval": "max",
        "fidelity": 1
    })

    history = r.json().get("history", [])

    for point in history:
        all_price_history.append({
            "conditionId":    row["conditionId"],
            "question":       row["question"],
            "market_start":   row["startDate"],
            "market_end":     row["endDate"],
            "result_yes_won": row["result_yes_won"],
            "timestamp":      pd.to_datetime(point["t"], unit="s", utc=True),
            "yes_price":      point["p"]
        })

    print(f"{row['question'][:50]}: {len(history)} satır")
    time.sleep(0.3)

df_prices = pd.DataFrame(all_price_history).sort_values("timestamp").reset_index(drop=True)
print(f"\nToplam fiyat noktası: {len(df_prices)}")

Bitcoin above 78,000 on May 4, 3AM ET?: 12 satır
Bitcoin above 79,200 on May 4, 3AM ET?: 13 satır
Bitcoin above 81,600 on May 4, 3AM ET?: 13 satır
Bitcoin above 80,400 on May 4, 3AM ET?: 13 satır
Bitcoin above 78,800 on May 4, 3AM ET?: 12 satır
Bitcoin above 80,800 on May 4, 3AM ET?: 13 satır
Bitcoin above 79,600 on May 4, 3AM ET?: 13 satır
Bitcoin above 81,200 on May 4, 3AM ET?: 13 satır
Bitcoin above 82,000 on May 4, 3AM ET?: 13 satır
Bitcoin above 78,400 on May 4, 3AM ET?: 13 satır
Bitcoin above 80,800 on May 4, 1AM ET?: 12 satır
Bitcoin above 81,600 on May 4, 1AM ET?: 12 satır
Bitcoin above 82,000 on May 4, 1AM ET?: 12 satır
Bitcoin above 79,200 on May 4, 1AM ET?: 12 satır
Bitcoin above 79,600 on May 4, 1AM ET?: 12 satır
Bitcoin above 81,200 on May 4, 1AM ET?: 12 satır
Bitcoin above 78,800 on May 4, 1AM ET?: 12 satır
Bitcoin above 78,000 on May 4, 1AM ET?: 12 satır
Bitcoin above 80,400 on May 4, 1AM ET?: 12 satır
Bitcoin above 78,400 on May 4, 1AM ET?: 12 satır
Bitcoin above 77,200

In [ ]:
exchange = ccxt.kraken()

# Use Polymarket time range — no hardcoded lookback
start_ts    = int(df_prices["timestamp"].min().timestamp() * 1000)
end_ts      = int(df_prices["timestamp"].max().timestamp() * 1000)
fetch_since = start_ts
all_ohlcv   = []

for _ in range(500):
    ohlcv = exchange.fetch_ohlcv("BTC/USDT", timeframe="1m", since=fetch_since, limit=1000)
    if not ohlcv:
        break
    all_ohlcv.extend(ohlcv)
    fetch_since = ohlcv[-1][0] + 60000
    if ohlcv[-1][0] >= end_ts:
        break
    time.sleep(0.3)

df_binance = pd.DataFrame(all_ohlcv, columns=["timestamp","open","high","low","close","volume"])
df_binance["timestamp"] = pd.to_datetime(df_binance["timestamp"], unit="ms", utc=True)
df_binance = df_binance.drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)

ts_min = df_binance["timestamp"].min()
ts_max = df_binance["timestamp"].max()
print(f"Kraken rows: {len(df_binance)}")
print(f"Date range : {ts_min} -> {ts_max}")
print(f"Poly range : {df_prices['timestamp'].min()} -> {df_prices['timestamp'].max()}")


In [19]:
df_merged = pd.merge_asof(
    df_prices,
    df_binance[["timestamp", "close", "volume"]],
    on="timestamp",
    direction="nearest",
    tolerance=pd.Timedelta("2min")
)
df_merged = df_merged.rename(columns={"close": "btc_price", "volume": "btc_volume"})

print(f"Merged satır: {len(df_merged)}")
print(f"NaN btc_price: {df_merged['btc_price'].isna().sum()}")

Merged satır: 558
NaN btc_price: 350


In [ ]:
os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

prev_count = 0
try:
    df_existing = pd.read_csv("data/processed/merged_dataset.csv")
    df_existing["timestamp"] = pd.to_datetime(df_existing["timestamp"], utc=True)
    prev_count   = len(df_existing)
    df_combined  = pd.concat([df_existing, df_merged], ignore_index=True)
    df_combined  = df_combined.drop_duplicates(subset=["conditionId", "timestamp"])
    df_combined  = df_combined.sort_values("timestamp").reset_index(drop=True)
    print(f"Previous rows: {prev_count} -> New total: {len(df_combined)}")
except FileNotFoundError:
    df_combined = df_merged
    print(f"First run: {len(df_combined)} rows")

df_combined.to_csv("data/processed/merged_dataset.csv", index=False)
df_prices.to_csv("data/raw/polymarket_raw.csv",         index=False)
df_binance.to_csv("data/raw/kraken_raw.csv",            index=False)

added = len(df_combined) - prev_count
print(f"Rows added this run : {added}")
print(f"Total accumulated   : {len(df_combined)}")
print("Saved.")


In [21]:
!git config user.email "kadirnsy@gmail.com"
!git config user.name "llelus"
!git add data/
!git commit -m "add: gunluk veri guncellendi"
!git push

[main c58900b] add: gunluk veri guncellendi
 3 files changed, 2257 insertions(+), 1203 deletions(-)
 rewrite data/raw/kraken_raw.csv (98%)
 rewrite data/raw/polymarket_raw.csv (99%)
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 30.36 KiB | 1.60 MiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/llelus/DSA-Project.git
   b145a18..c58900b  main -> main


In [22]:
df_combined = pd.read_csv("data/processed/merged_dataset.csv")
df_combined["timestamp"] = pd.to_datetime(df_combined["timestamp"], utc=True)

print(f"Toplam satır: {len(df_combined)}")
print(f"Tarih aralığı: {df_combined['timestamp'].min()} → {df_combined['timestamp'].max()}")
print(f"Unique market: {df_combined['conditionId'].nunique()}")
print(f"NaN btc_price: {df_combined['btc_price'].isna().sum()}")
print(f"\nÖrnek:")
print(df_combined[["timestamp","question","yes_price","btc_price"]].head(5).to_string())

Toplam satır: 4776
Tarih aralığı: 2026-04-09 19:00:42+00:00 → 2026-05-04 08:00:06+00:00
Unique market: 278
NaN btc_price: 350

Örnek:
                  timestamp                                  question  yes_price  btc_price
0 2026-04-09 19:00:42+00:00  Bitcoin above 71,600 on April 9, 4PM ET?      0.525    71900.0
1 2026-04-09 19:00:44+00:00  Bitcoin above 74,000 on April 9, 4PM ET?      0.445    71900.0
2 2026-04-09 19:00:53+00:00  Bitcoin above 72,400 on April 9, 4PM ET?      0.455    71900.0
3 2026-04-09 19:00:54+00:00  Bitcoin above 72,800 on April 9, 4PM ET?      0.450    71900.0
4 2026-04-09 19:00:55+00:00  Bitcoin above 73,200 on April 9, 4PM ET?      0.450    71900.0
